In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient I: improving a policy directly

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position: Part B (policy gradient), session 1 of 4**
    - Part A (the value-based path) learned values and read the policy off them by an argmax; it ended at the limitations of that argmax
    - Part B puts the network on the policy itself; the stochastic-policy methods of this part are built on **one theorem**, stated in this session
    - **the limitation we start from:** the argmax needs a short list of actions and gives greedy (deterministic) policies
- **The question:** how do we improve a policy directly, when the environment's response is unknown and so cannot be differentiated through?
- **By the end you can**
    - say what a policy network outputs, for discrete and for continuous actions
    - write the objective $J(\theta)$ and say why its gradient is not obvious
    - state the policy gradient theorem in its two forms and say why the transition probabilities $\transp$ do not appear
    - say what one update does to the action probabilities
    - say why this estimator is on-policy
    - say where the proofs are, and why to read them


**Where this session sits in Part B**

<img src="images/partB_arc_b1.png" width=95%>


# A policy as a network

**Why this section?** Part A ended by putting the network on the policy instead of on the values. First, what does such a network output?

- the policy $\pi_\theta(\act \mid \state) = \pr{\actseq_\tt = \act \mid \stateseq_\tt = \state}$ is a *probability distribution* over actions, given the state (a *density* when actions are continuous)
    - $\theta$ are the network's parameters (its weights)
- the network outputs the numbers that define the distribution; the agent **samples** its action from it
    - **discrete actions:** one score (logit) per action, turned into probabilities by a **softmax** (exponentiate the scores and divide by their sum)
    - **continuous actions:** the network outputs the mean and spread of a Gaussian (a bell-shaped distribution), and the action is sampled from it
    - Microduck's policy is of the second kind: continuous motor targets
- randomness is built into the policy, so exploration needs no $\epsilon$-greedy rule (the spread may shrink as learning proceeds)

<img src="images/policy_heads.png" width=90%>


Two sources of randomness

- the **policy** is random: it samples $\act$ from $\pi_\theta(\cdot \mid \state)$
- the **environment** can also be random: $\transp(\state', \rew \mid \state, \act)$
    - a single observed $(\state', \rew)$ is only a noisy sample of that distribution
    - every model-free method must work from such samples
- the theorem of this session will show that the gradient never needs $\transp$ itself


## Improving the policy: gradient ascent on $J(\theta)$

**Why this section?** With a parameterized policy, "improving the policy" means changing $\theta$. That needs a number to improve and a direction to move.

- policy gradient methods create a sequence of policies $\pi_0, \ldots, \pi_p, \ldots$ by a sequence of parameters $\theta_0, \ldots, \theta_p, \ldots$
- each step is **gradient ascent** on an objective $J(\theta)$ (ascent, not descent, because we *maximize*; we add the gradient where a loss would subtract it)

$$
\theta_{p+1} = \theta_p + \alpha \, \nabla_\theta J(\theta_p)
$$

- $J(\theta)$ is the **objective**: a number saying how good $\pi_\theta$ is
- $\nabla_\theta J(\theta)$ is its gradient with respect to the parameters $\theta$
- $\alpha$ is the step size (learning rate)
- source for the chapter: Sutton and Barto, *Reinforcement Learning*, Chapter 13


## The objective: expected return

**Why this section?** Gradient ascent needs something to maximize: the expected return.

- $J(\theta)$ is the *expected value*, across possible episodes $\tau$, of the return $\rlret_{0,\tau}$ from the initial state

$$
J(\theta) = \Exp{\tau \sim \pi_\theta} ( \rlret_{0, \tau} )
$$

- $\tau \sim \pi_\theta$ means episodes generated by following $\pi_\theta$; their probability $\prc{\tau}{\theta}$ depends on $\theta$
- this is the value of the initial state, $\statevalfun_{\pi}(\stateseq_0)$, from Part A (averaged over the distribution of starting states, if it is not fixed)

Recall the return from step $\tt$ of an episode

$$
\rlret_{\tt, \tau} = \sum_{k=0}^{|\tau|-\tt-1} \disc^k \, \rewseq_{\tt+k+1} = \rewseq_{\tt+1} + \disc \, \rlret_{\tt+1, \tau}
$$


## Why the gradient is not obvious

**Why this section?** $J$ is an expectation over episodes, and the environment's probabilities are unknown. How can we get its gradient at all?

- we need $\nabla_\theta J(\theta_p)$
- write the expectation as a probability-weighted sum over episodes

$$
\Exp{\tau \sim \pi_\theta} ( \rlret_{0, \tau} ) = \sum_\tau \prc{\tau}{\theta} \, \rlret_{0, \tau}
$$

- the probability of an episode chains the policy and the environment at every step

$$
\prc{\tau}{\theta} = \pr{\stateseq_{\tau,0}} \prod_{\tt=0}^{|\tau|-1} \pi_\theta(\actseq_{\tau,\tt} \mid \stateseq_{\tau,\tt}) \; \transp(\stateseq_{\tau, \tt+1}, \rewseq_{\tau,\tt+1} \mid \stateseq_{\tau,\tt}, \actseq_{\tau,\tt})
$$

- the $\transp$ factors (and the start distribution) are the environment's response, **not available** in a model-free setting
- so: how do we differentiate an expectation when we do not know the distribution?

<img src="images/b1_trajectory_chain.png" width=90%>


# The policy gradient theorem

**Why this section?** It answers the question just posed: the gradient can be written without the environment's probabilities.

- **Result:** $\nabla_\theta J(\theta)$ is an expectation, over episodes from $\pi_\theta$, of $\nabla_\theta \log \pi_\theta$ weighted by a return
- the transition probabilities $\transp$ **do not appear**
    - why, in one line: the log of a product is a sum, and the $\transp$ factors do not depend on $\theta$, so their gradient is $0$ (details: the appendix)
- conditions: episodic tasks; $\pi_\theta$ differentiable in $\theta$; the start distribution and $\transp$ do not depend on $\theta$
- so the gradient can be *estimated from sampled episodes*, without knowing $\transp$

**Episode-return form**: every step's term is weighted by the whole episode's return

$$
\nabla_\theta J(\theta) = \Exp{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tau,\tt} \mid \stateseq_{\tau,\tt}) \; \rlret_{0,\tau} \right]
$$

- natural when the reward comes at the end of the episode (a single reward)


**Return-to-go form**: each step's term is weighted only by the return from that step on

$$
\nabla_\theta J(\theta) = \E_{\tau \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\tau|-1} \nabla_\theta \log \pi_\theta(\actseq_{\tau,\tt} \mid \stateseq_{\tau,\tt}) \; \disc^\tt \, \rlret_{\tt,\tau} \right]
$$

- natural when rewards arrive at intermediate steps; an action cannot affect rewards that came before it
- the two forms are equal *in expectation* (path by path they differ, except for a single terminal reward); the factor $\disc^\tt$ is $1$ when $\disc = 1$
- many implementations drop $\disc^\tt$; the gradient they compute is then a different (biased) estimate for the discounted objective

Either way
- this theorem is the foundation of the stochastic-policy gradient methods of this part (deterministic-policy methods rest on a different result)
- $\nabla_\theta \log \pi_\theta(\act \mid \state)$ (called the **score function**) is the direction in parameter space that makes action $\act$ more likely in state $\state$
- the weight in front of it (here a return) says *how much* to push


**What one update does** (one state, a softmax policy over three actions)

- the parameters $\theta$ are the three *logits*, here $\theta = (0.4, 0, -0.4)$, and the second action is the one sampled; for a softmax over logits, $\nabla_\theta \log \pi_\theta(\act) = \text{onehot}(\act) - \pi_\theta$ (for a network, apply the chain rule through the logits)
- the update is $\theta \leftarrow \theta + \alpha \, \rlret \, \nabla_\theta \log \pi_\theta(\act)$, with $\alpha = 0.5$ and $\rlret$ the return that followed the sampled action $\act$ (the figure uses $\rlret = +1$ and $-1$ as illustrations)

<img src="images/policy_gradient_bars.png" width=95%>

- an action followed by a *positive* return becomes more likely, one followed by a *negative* return less likely; the other actions move the opposite way (the probabilities sum to $1$)
- if every return is positive, every sampled action is pushed up: the *expected* gradient is the same whether or not a constant is subtracted from the returns (the next session shows why), but the sampled estimate is very noisy (high *variance*, as defined with Monte Carlo targets); fixing that is the next session


## From the theorem to an algorithm: sample episodes

**Why this section?** The theorem is an expectation over episodes. We cannot sum over all episodes, but we can sample some.

- run the current policy $\pi_\theta$ to generate $N$ episodes, and replace the expectation by an average

$$
\widehat{\nabla_\theta J}(\theta) = \frac{1}{N} \sum_{i=1}^{N} \sum_{\tt=0}^{|\tau_i|-1} \nabla_\theta \log \pi_\theta(\actseq_{i,\tt} \mid \stateseq_{i,\tt}) \, \rlret_{0,\tau_i}
$$

- then take a step $\theta \leftarrow \theta + \alpha \, \widehat{\nabla_\theta J}(\theta)$, where the hat marks the sampled *estimate*
- this estimator is the core of **REINFORCE** (the next session)
- the environment's effect enters only through the sampled rewards and next states: the model $\transp$ is never needed
- it is **on-policy**: the episodes must come from the policy being improved, so after each step the old episodes are used up
    - contrast deep Q-learning, which reused old transitions through replay because Q-learning is off-policy
    - PPO's probability ratio will later let one batch of episodes be reused for several updates (session 3)


## Read the proof: the appendix

**Why this section?** The theorem is surprising: the environment drops out. Its proof takes a few slides, and its tools are used next session.

- **We strongly recommend reading the appendix** (`RL_B1_Appendix_PG_proofs.ipynb`) before the next session: the baseline of the next session is a two-line use of its zero-mean fact
- the two ideas in it
    - **the log-derivative trick** (also called the likelihood-ratio trick): $\nabla_\theta p_\theta(x) = p_\theta(x)\, \nabla_\theta \log p_\theta(x)$, which turns the gradient of an expectation into an expectation of a gradient
    - **the log of an episode's probability is a sum of logs**; the $\transp$ terms do not depend on $\theta$, so their gradient is zero and they vanish
- the appendix also holds
    - why the return-to-go form equals the episode-return form (the *score function* has zero mean)
    - an alternate proof from per-step rewards
- **where it is used**
    - the zero-mean fact justifies subtracting a baseline in the next session (the baseline must not depend on the action)
    - later sessions refer back to the log-derivative form $\nabla_\theta \pi_\theta = \pi_\theta \nabla_\theta \log \pi_\theta$
- the main line does not need the proofs to follow the next steps, but without them the theorem is taken on trust


## Preview: one theorem, three slots

**Why this section?** The estimator just built is one formula, and the plan for Part B changes only parts of it. Naming the parts now shows where each later session fits (a plan, to be confirmed as we go).

$$
\theta \leftarrow \theta + \underbrace{\alpha}_{\text{slot 2: how far}} \; \text{average over sampled steps of} \;\; \underbrace{w}_{\text{slot 1: weight}} \; \nabla_\theta \log \pi_\theta(\act \mid \state)
$$

| Slot | The question | Planned |
|:--|:--|:--|
| **1. weight** $w$ on each gradient term | the raw return is very noisy: what should weight each step? (baseline, advantage; a learned value *can* help: *actor-critic*) | next session |
| **2. how far** one update moves the policy | how far before the estimate stops being trustworthy? (PPO also changes the objective, using a ratio of probabilities) | session 3 |
| **3. source of the baseline** inside slot 1 | can the baseline come without training a value network at all? (GRPO) | session 4 |

- the starting point stays fixed (the theorem); the methods (REINFORCE, actor-critic, PPO, GRPO) differ in how they fill these slots
- slot 3 sits inside the weight $w$ of slot 1, so the formula shows two places; the table is a preview now; it is filled in session by session and closed in session 4
- *actor-critic* is a **use** of a value model (the value-based machinery of Part A) inside this part; it does not turn Part B into value-based learning


## Aside: methods that learn a policy without a gradient

- Policy iteration (Part A) also improves a policy: it alternates evaluating a value function and improving the policy by an argmax
- so it is both value-based and policy-improving, but it does not move the policy by gradients
- other gradient-free policy search methods exist; this part covers only policy gradient


# What we established, and what it sets up

- **Established**
    - a policy network outputs a distribution (softmax for discrete actions, mean and spread for continuous); the agent samples from it, so exploration is built in
    - $J(\theta)$ is the expected return; we climb it by gradient ascent
    - the gradient is an expectation over episodes of $\nabla_\theta \log \pi_\theta$ times a return, and the transition probabilities $\transp$ do not appear: it can be estimated from sampled episodes
    - the two forms of the theorem (episode return, return-to-go) agree in expectation
    - the estimator is **on-policy**
    - the proofs, including the log-derivative trick, are in the appendix (recommended)
- **Sets up**
    - next session: the raw return is a noisy weight (high variance); baselines and advantages reduce it, and a learned value can supply the baseline (actor-critic)
    - then how far one update may move (PPO), and a baseline without a critic (GRPO)
